## 📊 Daily Odin Global User Report Generator (VS Code Edition) — **v1**

This interactive notebook wraps **`daily_odin_report.py`** and breaks it into easy‑to‑test units so you can:

* configure variables in one place  
* run each functional step independently  
* experiment with different report windows, batch sizes, etc.  

> **Important**: **Upload and import the script first** (see next section), then edit the configuration.


## 🗃️ Process & Aggregate User Report Data

In [ ]:
import sys

# @@ Install ipykernel
get_ipython().system(f'{sys.executable} -m ipykernel install --user --name=odin_env --display-name="Odin Environment"')  # pyright: ignore[reportUndefinedVariable]

# @@ Install / upgrade required libraries (Colab already has pandas/requests) @@
get_ipython().system(f'{sys.executable} -m pip install paramiko python-dateutil tqdm pandas requests ipywidgets')  # pyright: ignore[reportUndefinedVariable]

## Configuration

Edit the cell below to set **environment variables** or direct constants that the script relies on.


In [ ]:
# @title Configuration {"run":"auto"}

# @@ Runtime settings (edit freely) @@
import os, json
from dataclasses import asdict

# --- Minimum required ---
ODIN_API_BASE_URL = ""
ODIN_API_USERNAME = ""
ODIN_API_PASSWORD = ""

SFTP_HOST = ""
SFTP_USERNAME = ""
SFTP_PASSWORD = ""
SFTP_REMOTE_PATH = ""

SMTP_HOST = ""
SMTP_USERNAME = ""
SMTP_PASSWORD = ""
SMTP_FROM = ""
SMTP_TO = ""
SMTP_PORT = ""


# --- Minimum required ---
os.environ['ODIN_API_BASE_URL'] = ODIN_API_BASE_URL
os.environ['ODIN_API_USERNAME'] = ODIN_API_USERNAME
os.environ['ODIN_API_PASSWORD'] = ODIN_API_PASSWORD

# --- Optional overrides ---
os.environ['BATCH_SIZE'] = '200'

# SFTP (leave blank to disable)
os.environ['SFTP_HOST'] = SFTP_HOST
os.environ['SFTP_USERNAME'] = SFTP_USERNAME
os.environ['SFTP_PASSWORD'] = SFTP_PASSWORD

# SMTP (leave blank to disable)
os.environ['SMTP_HOST'] = SMTP_HOST
os.environ['SMTP_PORT'] = SMTP_PORT
os.environ['SMTP_USERNAME'] = SMTP_USERNAME
os.environ['SMTP_PASSWORD'] = SMTP_PASSWORD
os.environ['SMTP_FROM'] = SMTP_FROM
os.environ['SMTP_TO'] = SMTP_TO
os.environ['SFTP_REMOTE_PATH'] = SFTP_REMOTE_PATH

print("✅ Configuration Settings Loaded")

## 📥 Load `daily_odin_report.py` local

In [ ]:
# === Load daily_odin_report.py from local filesystem (no upload required) ===
import os, importlib.util, sys

path = "daily_odin_report.py"
assert os.path.exists(path), f"{path} not found in current directory!"

# Dynamic import
spec = importlib.util.spec_from_file_location("daily_odin_report", path)
dcr  = importlib.util.module_from_spec(spec)
sys.modules["daily_odin_report"] = dcr
spec.loader.exec_module(dcr)

print("✅ daily_odin_report.py loaded from local filesystem and imported as 'dcr'")

## 🔑 Authenticate & Create API Client

In [ ]:
cfg = dcr.Config()
print(json.dumps(asdict(cfg), indent=2))
cfg = dcr.Config()                       # pick up env vars
api_client = dcr.OdinAPIClient(cfg)
assert api_client.authenticate(), "API authentication failed ❌"
print("Authenticated ✔")

## 🌍 Fetch Global User Report for Service Providers

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

# Fetch comprehensive global user data from ALL service providers
print("Fetching global user report from all service providers...")
global_users = api_client.get_global_user_report()

print(f"\nTotal users in global report: {len(global_users):,}")

# Display sample global user data
if global_users:
    print("\nSample global user data:")
    sample_user = global_users[0]
    for key, value in list(sample_user.items())[:10]:  # Show first 10 fields
        print(f"  {key}: {value}")
    
    # Convert to DataFrame for easier viewing
    df_global_users = pd.DataFrame(global_users)
    print(f"\nDataFrame shape: {df_global_users.shape}")
    print("\nColumns:", list(df_global_users.columns))
    
    # Show summary by service provider
    if 'serviceProviderId' in df_global_users.columns:
        sp_summary = df_global_users.groupby('serviceProviderId').size()
        print("\nUsers per Service Provider (Global Report):")
        for sp_id, count in sp_summary.items():
            print(f"  {sp_id}: {count} users")
        
        # Show total unique service providers
        unique_sps = df_global_users['serviceProviderId'].nunique()
        print(f"\nTotal unique Service Providers: {unique_sps}")
        
        # Show unique groups if available
        if 'groupId' in df_global_users.columns:
            unique_groups = df_global_users['groupId'].nunique()
            print(f"Total unique Groups: {unique_groups}")
    
    # Optional: Process with GlobalUserDataProcessor
    PROCESS_GLOBAL_DATA = True # @param {"type":"boolean"}
    
    if PROCESS_GLOBAL_DATA:
        print("\nProcessing global user data...")
        processor = dcr.GlobalUserDataProcessor(include_optional_fields=True)
        df_clean, df_summary = processor.process_user_data(global_users)
        
        print(f"Processed data shape: {df_clean.shape}")
        print(f"Summary data shape: {df_summary.shape}")
        
        if not df_summary.empty:
            print("\nSummary by Service Provider:")
            print(df_summary.head())
else:
    print("No global user data found")

## 🗃️ Process & Aggregate Global User Report

In [ ]:
## 🌍 Process & Aggregate Global User Report

# If you have run the "Fetch Global User Report for Service Providers" cell and have df_global_users:
if 'df_global_users' in globals():
    print("Processing and aggregating global user report data...")
    processor = dcr.GlobalUserDataProcessor(include_optional_fields=True)
    df_global_clean, df_global_summary = processor.process_user_data(df_global_users.to_dict(orient='records'))
    print(f"Processed global user data shape: {df_global_clean.shape}")
    print(f"Global user summary data shape: {df_global_summary.shape}")
    if not df_global_summary.empty:
        print("\nGlobal User Summary by Service Provider:")
        print(df_global_summary.head())
else:
    print("No global user report data found. Please run the global user report fetch cell first.")

## 💾 Export Global User Report to CSV

In [ ]:
## 💾 Export Global User Report to CSV

# If you have run the "Process & Aggregate Global User Report" cell and have df_global_clean and df_global_summary:
if 'df_global_clean' in globals() and 'df_global_summary' in globals():
    exporter = dcr.ReportExporter(cfg)
    global_raw_csv, global_summary_csv = exporter.export_user_data_to_csv(df_global_clean, df_global_summary)
    print('Global User Report Files:', global_raw_csv, global_summary_csv)
else:
    print("No processed global user report data found. Please run the global user report processing cell first.")

## ☁️ Optional Outputs (SFTP Upload & Email)

In [ ]:
## ☁️ Optional Outputs (SFTP Upload & Email)

uploaded = False; emailed = False
if 'exporter' in globals() and 'global_raw_csv' in globals() and 'global_summary_csv' in globals():
    if cfg.sftp_host:
        uploaded = exporter.upload_to_sftp([global_raw_csv, global_summary_csv])
    if cfg.smtp_host:
        # Optionally, you can compute a summary or pass an empty dict
        emailed = exporter.send_email_report([global_raw_csv, global_summary_csv], {})
    print('SFTP:', uploaded, 'Email:', emailed)
else:
    print("No exported global user report files found. Please run the export cell first.")

---
### ✅ Notebook Complete
Generated 2025-06-27 21:03 UTC